# 🧠 Self-Diagnosing Neural Network (SDNN)
## Reliable Image Classification for Autonomous Perception Systems

This notebook runs the complete SDNN pipeline end-to-end:
1. Environment setup
2. CIFAR-10 dataset loading
3. SDNN training (with auxiliary confidence + error branches)
4. Evaluation (Accuracy, ECE, NLL, Brier Score, AUROC)
5. Reliability diagram generation
6. Baseline comparison (Standard CNN + Temperature Scaling)

> **Runtime**: Enable GPU → Runtime > Change runtime type > T4 GPU

---
## 1. Setup

In [ ]:
# Install / verify dependencies
!pip install torch torchvision numpy scikit-learn matplotlib seaborn tqdm --quiet
print('Dependencies ready')

In [ ]:
import os

# ── Option A: Clone from GitHub (update URL to your repo) ──────────────────
# !git clone https://github.com/YOUR_USERNAME/sdnn_project.git
# os.chdir('sdnn_project')

# ── Option B: Upload the sdnn_project/ folder to Colab, then set path ──────
# from google.colab import files
# uploaded = files.upload()   # upload as .zip, then unzip below
# !unzip sdnn_project.zip
# os.chdir('sdnn_project')

# ── If already in sdnn_project/ root: ──────────────────────────────────────
import sys
PROJECT_ROOT = os.path.abspath('.')   # adjust if needed
sys.path.insert(0, PROJECT_ROOT)
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Device
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f'PyTorch  : {torch.__version__}')
print(f'Device   : {device}')
if device.type == 'cuda':
    print(f'GPU      : {torch.cuda.get_device_name(0)}')

---
## 2. CIFAR-10 Data Loading

CIFAR-10 contains 10 classes, all relevant to the autonomous perception framing:

| Class | Autonomous Context |
|-------|-------------------|
| automobile, truck | Vehicle detection |
| airplane, ship | Extended environment |
| deer, horse, bird | Road hazard detection |
| cat, dog, frog | Pedestrian-adjacent obstacles |

In [ ]:
from training.train import build_dataloaders

BATCH_SIZE = 64

train_loader, test_loader = build_dataloaders(batch_size=BATCH_SIZE, num_workers=2)
print(f'Train batches : {len(train_loader)}  ({len(train_loader.dataset):,} images)')
print(f'Test  batches : {len(test_loader)}  ({len(test_loader.dataset):,} images)')

In [ ]:
from models.sdnn_model import CIFAR10_CLASSES
import torchvision

# Visualise a batch of training images
images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
fig.suptitle('CIFAR-10 Training Samples', fontsize=13, fontweight='bold')

for i, ax in enumerate(axes.flat):
    img = images[i].permute(1, 2, 0).numpy()
    img = img * 0.5 + 0.5   # denormalize
    ax.imshow(img.clip(0, 1))
    ax.set_title(CIFAR10_CLASSES[labels[i]], fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.show()

---
## 3. SDNN Architecture

```
Input (3×32×32)
    │
    ▼
CIFARResNet18Backbone → 512-D feature vector
    │
  ┌─┼──────────┐
  ▼ ▼          ▼
Class  Confidence  Error Prediction
Head    Head        Head
  │       │              │
logits  conf (0-1)   err_prob (0-1)
```

**Loss:**  `CrossEntropy + 0.5×BCE(confidence) + 0.5×BCE(error_prob)`

In [ ]:
from models.sdnn_model import SDNN
from training.loss_functions import SDNNLoss

model   = SDNN(num_classes=10).to(device)
loss_fn = SDNNLoss(lambda1=0.5, lambda2=0.5)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Total parameters    : {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

# Quick shape check
dummy = torch.randn(2, 3, 32, 32).to(device)
out   = model(dummy)
print(f"\nForward pass shapes:")
print(f"  logits     : {out['logits'].shape}")
print(f"  confidence : {out['confidence'].shape}")
print(f"  error_prob : {out['error_prob'].shape}")

---
## 4. Training

In [ ]:
# ── Hyperparameters ──────────────────────────────────────────────────────
EPOCHS    = 50      # Increase to 100 on A100 for best results
LR        = 1e-3
LAMBDA1   = 0.5     # weight for confidence calibration loss
LAMBDA2   = 0.5     # weight for error prediction loss
SAVE_DIR  = 'experiments/'
os.makedirs(SAVE_DIR, exist_ok=True)

print(f'Epochs  : {EPOCHS}')
print(f'LR      : {LR}')
print(f'Lambda1 : {LAMBDA1}  (confidence loss weight)')
print(f'Lambda2 : {LAMBDA2}  (error prediction loss weight)')

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau

optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5, verbose=True)

In [ ]:
from training.train import train_one_epoch, evaluate
import time

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_loss = float('inf')

print(f"{'Epoch':>6} {'Train Loss':>12} {'Train Acc':>10} {'Val Loss':>10} {'Val Acc':>9} {'Time':>7}")
print("-" * 62)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, loss_fn, device)
    val_metrics           = evaluate(model, test_loader, loss_fn, device)
    val_loss, val_acc     = val_metrics['loss'], val_metrics['accuracy']

    scheduler.step(val_loss)
    elapsed = time.time() - t0

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(f"{epoch:>6} {train_loss:>12.4f} {train_acc:>10.4f} "
          f"{val_loss:>10.4f} {val_acc:>9.4f} {elapsed:>6.1f}s")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch, 'model_state': model.state_dict(),
            'val_loss': val_loss, 'val_acc': val_acc,
        }, os.path.join(SAVE_DIR, 'best_sdnn.pth'))
        print(f"         ↳ ✓ Best checkpoint saved (val_loss={val_loss:.4f})")

print(f'\nTraining complete. Best val loss: {best_val_loss:.4f}')

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('SDNN Training History', fontsize=13, fontweight='bold')

epochs_range = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs_range, history['train_loss'], label='Train Loss', color='#3498db')
axes[0].plot(epochs_range, history['val_loss'],   label='Val Loss',   color='#e74c3c')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Curve'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_range, history['train_acc'], label='Train Acc', color='#3498db')
axes[1].plot(epochs_range, history['val_acc'],   label='Val Acc',   color='#e74c3c')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy Curve'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: experiments/training_curves.png')

---
## 5. Evaluation

We evaluate with 5 reliability-aware metrics:

| Metric | Purpose | Goal |
|--------|---------|------|
| Accuracy | Classification correctness | ≥ 88% |
| ECE | Confidence calibration | ≤ 0.05 |
| NLL | Quality of probability estimates | lower |
| Brier Score | Probability accuracy | lower |
| AUROC | Ability to detect own errors | ≥ 0.70 |

In [ ]:
# Load best checkpoint for evaluation
ckpt = torch.load(os.path.join(SAVE_DIR, 'best_sdnn.pth'), map_location=device)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f"Loaded checkpoint from epoch {ckpt['epoch']} (val_loss={ckpt['val_loss']:.4f}, val_acc={ckpt['val_acc']:.4f})")

In [ ]:
# Collect predictions on test set
all_softmax, all_confidence, all_error_prob, all_labels = [], [], [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        out    = model(images)

        softmax = torch.softmax(out['logits'], dim=1)
        all_softmax.append(softmax.cpu().numpy())
        all_confidence.append(out['confidence'].squeeze(1).cpu().numpy())
        all_error_prob.append(out['error_prob'].squeeze(1).cpu().numpy())
        all_labels.append(labels.numpy())

softmax_probs = np.concatenate(all_softmax)
confidence    = np.concatenate(all_confidence)
error_prob    = np.concatenate(all_error_prob)
labels_np     = np.concatenate(all_labels)

print(f'Collected {len(labels_np):,} test predictions')

In [ ]:
from evaluation.metrics import compute_all_metrics, print_metrics

# Compute all SDNN metrics
sdnn_metrics = compute_all_metrics(softmax_probs, confidence, error_prob, labels_np)
print_metrics(sdnn_metrics, title='SDNN — Autonomous Perception on CIFAR-10')

---
## 6. Reliability Diagram

A perfectly calibrated model lies on the diagonal (confidence = accuracy).  
- **Red bars** = overconfident (model is more confident than accurate)  
- **Blue bars** = underconfident

In [ ]:
from evaluation.reliability_diagram import plot_reliability_diagram

predicted = softmax_probs.argmax(axis=1)
correct   = (predicted == labels_np)

plot_reliability_diagram(
    confidence=confidence,
    correct=correct,
    n_bins=15,
    save_path=os.path.join(SAVE_DIR, 'sdnn_reliability_diagram.png'),
    model_label='SDNN',
)
plt.show()

---
## 7. Baseline Comparison

We compare SDNN against two baselines:
- **Standard CNN**: same backbone, no auxiliary branches
- **Temperature Scaling**: post-hoc calibration on Standard CNN

In [ ]:
import torch.nn as nn
from training.train_baseline import StandardCNN, TemperatureScaling, train_standard_cnn
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Train Standard CNN baseline
print('=' * 60)
print('Training Baseline: Standard CNN')
print('=' * 60)

std_cnn   = StandardCNN(num_classes=10).to(device)
opt_base  = optim.Adam(std_cnn.parameters(), lr=LR)
sched_base = ReduceLROnPlateau(opt_base, patience=5, factor=0.5, verbose=True)

std_cnn = train_standard_cnn(
    std_cnn, train_loader, test_loader,
    opt_base, sched_base, device, EPOCHS
)

In [ ]:
from evaluation.metrics import compute_all_metrics, print_metrics

# Standard CNN metrics
std_softmax, std_labels = [], []
std_cnn.eval()
with torch.no_grad():
    for imgs, lbls in test_loader:
        logits = std_cnn(imgs.to(device))
        std_softmax.append(torch.softmax(logits, dim=1).cpu().numpy())
        std_labels.append(lbls.numpy())
std_softmax = np.concatenate(std_softmax)
std_labels  = np.concatenate(std_labels)
std_conf    = std_softmax.max(axis=1)
std_err     = 1.0 - std_conf

baseline_metrics = compute_all_metrics(std_softmax, std_conf, std_err, std_labels)
print_metrics(baseline_metrics, title='Baseline: Standard CNN')

# Temperature Scaling
ts_model = TemperatureScaling(std_cnn).to(device)
ts_model.fit(test_loader, device)

ts_softmax, ts_labels = [], []
ts_model.eval()
with torch.no_grad():
    for imgs, lbls in test_loader:
        logits = ts_model(imgs.to(device))
        ts_softmax.append(torch.softmax(logits, dim=1).cpu().numpy())
        ts_labels.append(lbls.numpy())
ts_softmax = np.concatenate(ts_softmax)
ts_labels  = np.concatenate(ts_labels)
ts_conf    = ts_softmax.max(axis=1)
ts_err     = 1.0 - ts_conf

ts_metrics = compute_all_metrics(ts_softmax, ts_conf, ts_err, ts_labels)
print_metrics(ts_metrics, title='Baseline: Temperature Scaling')

In [ ]:
# Summary comparison table
print('\n' + '═' * 72)
print('  FINAL COMPARISON')
print('═' * 72)
print(f"  {'Metric':<28} {'Std CNN':>10} {'Temp Scale':>12} {'SDNN':>8}")
print('-' * 72)

rows = [
    ('Accuracy      (↑)', 'accuracy',    '{:.4f}'),
    ('ECE           (↓)', 'ece',         '{:.4f}'),
    ('NLL           (↓)', 'nll',         '{:.4f}'),
    ('Brier Score   (↓)', 'brier_score', '{:.4f}'),
    ('AUROC Error   (↑)', 'auroc_error', '{:.4f}'),
]

for label, key, fmt in rows:
    v_std  = fmt.format(baseline_metrics[key])
    v_ts   = fmt.format(ts_metrics[key])
    v_sdnn = fmt.format(sdnn_metrics[key])
    print(f"  {label:<28} {v_std:>10} {v_ts:>12} {v_sdnn:>8}")

print('═' * 72)

In [ ]:
# Demonstrate the self-diagnosis capability
print('\n── Self-Diagnosis Demo (Autonomous Systems Context) ──')
print('Shows SDNN output for 10 random test images')
print()

test_images, test_labels = next(iter(test_loader))
model.eval()
with torch.no_grad():
    out = model(test_images[:10].to(device))

preds   = out['logits'].argmax(dim=1).cpu()
confs   = out['confidence'].squeeze(1).cpu()
errs    = out['error_prob'].squeeze(1).cpu()
actuals = test_labels[:10]

THRESHOLD = 0.3   # if error_prob > 0.3 → trigger safe fallback

print(f"  {'#':>2}  {'Actual':<12} {'Predicted':<12} {'Conf':>6} {'ErrProb':>8} {'Correct?':>9} {'Action':>20}")
print('  ' + '-' * 78)
for i in range(10):
    correct    = '✓' if preds[i] == actuals[i] else '✗'
    action     = '⚠ SAFE FALLBACK' if errs[i] > THRESHOLD else 'Proceed'
    print(f"  {i+1:>2}  {CIFAR10_CLASSES[actuals[i]]:<12} {CIFAR10_CLASSES[preds[i]]:<12} "
          f"{confs[i]:>6.3f} {errs[i]:>8.3f} {correct:>9}  {action:>20}")

In [ ]:
# Save all results as numpy arrays for further analysis
np.save(os.path.join(SAVE_DIR, 'sdnn_results.npy'), {
    'softmax_probs': softmax_probs,
    'confidence':    confidence,
    'error_prob':    error_prob,
    'labels':        labels_np,
    'sdnn_metrics':  sdnn_metrics,
    'baseline_metrics': baseline_metrics,
    'ts_metrics':    ts_metrics,
})

print('All results saved to experiments/')
print('  best_sdnn.pth                – trained SDNN weights')
print('  sdnn_reliability_diagram.png – calibration plot')
print('  training_curves.png          – loss & accuracy curves')
print('  sdnn_results.npy             – raw predictions & metrics')